In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Big Data Analysis") \
    .getOrCreate()

spark

In [ ]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/content/drive/MyDrive/yellow_tripdata_2015-01.csv")

In [ ]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RateCodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       2| 2015-01-15 19:05:39|  2015-01-15 19:23:42|              1|         1.59|  -73.993896484375|  40.7501106262207|         1|    

In [ ]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'pickup_longitude',
 'pickup_latitude',
 'RateCodeID',
 'store_and_fwd_flag',
 'dropoff_longitude',
 'dropoff_latitude',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount']

Select required columns

In [ ]:
df_ml = df.select(
    "passenger_count",
    "trip_distance",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "fare_amount"
)

df_ml.show(5)

+---------------+-------------+------------------+------------------+------------------+------------------+-----------+
|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|fare_amount|
+---------------+-------------+------------------+------------------+------------------+------------------+-----------+
|              1|         1.59|  -73.993896484375|  40.7501106262207|-73.97478485107422| 40.75061798095703|       12.0|
|              1|          3.3|-74.00164794921875|  40.7242431640625|-73.99441528320312| 40.75910949707031|       14.5|
|              1|          1.8|-73.96334075927734| 40.80278778076172|-73.95182037353516| 40.82441329956055|        9.5|
|              1|          0.5|-74.00908660888672| 40.71381759643555|-74.00432586669922| 40.71998596191406|        3.5|
|              1|          3.0|-73.97117614746094|40.762428283691406|-74.00418090820312|40.742652893066406|       15.0|
+---------------+-------------+---------

Data Cleaning

In [ ]:
# Remove null values
df_ml = df_ml.dropna()

In [ ]:
# Remove wrong values
df_ml = df_ml.filter(df_ml.trip_distance > 0)
df_ml = df_ml.filter(df_ml.fare_amount > 0)

Feature Engineering

In [ ]:
# Convert inputs into features
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "passenger_count",
        "trip_distance",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude"
    ],
    outputCol="features"
)

data = assembler.transform(df_ml)
data = data.select("features", "fare_amount")

In [ ]:
# Train-Test Split
train_data, test_data = data.randomSplit([0.8, 0.2])

In [ ]:
# Apply Model (Linear Regression)
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="fare_amount"
)

model = lr.fit(train_data)

In [ ]:
# Prediction
predictions = model.transform(test_data)

predictions.select("fare_amount", "prediction").show(5)

+-----------+------------------+
|fare_amount|        prediction|
+-----------+------------------+
|        2.5|11.992861988314914|
|        4.0|11.992861988314914|
|       52.0|11.992861988314914|
|       58.0|11.992861988314914|
|        2.5|11.992861994127368|
+-----------+------------------+
only showing top 5 rows


In [ ]:
# Model Evaluation
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator.evaluate(predictions)

print("RMSE:", rmse)

RMSE: 9.783452572210258


TASK 4: Sentiment Analysis

In [ ]:
data = [
    ("The ride was amazing and smooth",),
    ("Driver was rude and late",),
    ("Average experience nothing special",),
    ("Very comfortable and fast service",),
    ("Bad service and dirty car",)
]

columns = ["text"]

df_text = spark.createDataFrame(data, columns)
df_text.show()

+--------------------+
|                text|
+--------------------+
|The ride was amaz...|
|Driver was rude a...|
|Average experienc...|
|Very comfortable ...|
|Bad service and d...|
+--------------------+



In [ ]:
# Text Preprocessing
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(inputCol="text", outputCol="words")
df_words = tokenizer.transform(df_text)

df_words.show(truncate=False)

+----------------------------------+---------------------------------------+
|text                              |words                                  |
+----------------------------------+---------------------------------------+
|The ride was amazing and smooth   |[the, ride, was, amazing, and, smooth] |
|Driver was rude and late          |[driver, was, rude, and, late]         |
|Average experience nothing special|[average, experience, nothing, special]|
|Very comfortable and fast service |[very, comfortable, and, fast, service]|
|Bad service and dirty car         |[bad, service, and, dirty, car]        |
+----------------------------------+---------------------------------------+



In [ ]:
# Remove Stopwords
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
df_clean = remover.transform(df_words)

df_clean.show(truncate=False)

+----------------------------------+---------------------------------------+---------------------------------------+
|text                              |words                                  |filtered_words                         |
+----------------------------------+---------------------------------------+---------------------------------------+
|The ride was amazing and smooth   |[the, ride, was, amazing, and, smooth] |[ride, amazing, smooth]                |
|Driver was rude and late          |[driver, was, rude, and, late]         |[driver, rude, late]                   |
|Average experience nothing special|[average, experience, nothing, special]|[average, experience, nothing, special]|
|Very comfortable and fast service |[very, comfortable, and, fast, service]|[comfortable, fast, service]           |
|Bad service and dirty car         |[bad, service, and, dirty, car]        |[bad, service, dirty, car]             |
+----------------------------------+----------------------------

In [ ]:
# Convert Text → Numbers
from pyspark.ml.feature import HashingTF

hashingTF = HashingTF(inputCol="filtered_words", outputCol="features")
df_features = hashingTF.transform(df_clean)

df_features.select("text", "features").show(truncate=False)

+----------------------------------+------------------------------------------------------+
|text                              |features                                              |
+----------------------------------+------------------------------------------------------+
|The ride was amazing and smooth   |(262144,[79709,100314,199468],[1.0,1.0,1.0])          |
|Driver was rude and late          |(262144,[5835,68727,259019],[1.0,1.0,1.0])            |
|Average experience nothing special|(262144,[20575,55875,98221,116996],[1.0,1.0,1.0,1.0]) |
|Very comfortable and fast service |(262144,[43756,113503,219137],[1.0,1.0,1.0])          |
|Bad service and dirty car         |(262144,[43756,71578,133803,145380],[1.0,1.0,1.0,1.0])|
+----------------------------------+------------------------------------------------------+



In [ ]:
# Add Sentiment Labels , 1 = Positive, 0 = Negative

from pyspark.sql.functions import when

df_labeled = df_features.withColumn(
    "label",
    when(df_features.text.contains("amazing"), 1)
    .when(df_features.text.contains("comfortable"), 1)
    .when(df_features.text.contains("rude"), 0)
    .when(df_features.text.contains("bad"), 0)
    .otherwise(1)
)

df_labeled.select("text", "label").show()

+--------------------+-----+
|                text|label|
+--------------------+-----+
|The ride was amaz...|    1|
|Driver was rude a...|    0|
|Average experienc...|    1|
|Very comfortable ...|    1|
|Bad service and d...|    1|
+--------------------+-----+



In [ ]:
# Train Model (Logistic Regression)
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label")

model = lr.fit(df_labeled)

In [ ]:
# Prediction
predictions = model.transform(df_labeled)

predictions.select("text", "label", "prediction").show()

+--------------------+-----+----------+
|                text|label|prediction|
+--------------------+-----+----------+
|The ride was amaz...|    1|       1.0|
|Driver was rude a...|    0|       0.0|
|Average experienc...|    1|       1.0|
|Very comfortable ...|    1|       1.0|
|Bad service and d...|    1|       1.0|
+--------------------+-----+----------+

